# 03 - Traffic Density Generation

Convert per-frame vehicle counts from `data/processed/vehicle_counts.csv` into a time-series dataset for traffic prediction modelling:

1. Aggregate vehicle counts into fixed time intervals
2. Add temporal features (time of day, day of week)
3. Add lag features (t-1, t-2, t-3)
4. Apply smoothing / noise reduction
5. Compute rate of change, peak density, rolling mean, rolling standard deviation
6. Categorise traffic density into low / moderate / high

Output: `data/processed/traffic_density.csv`

**Split:** S01, S03, S04 = train · S02 = validation · S05 = test

In [1]:
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
processed_dir = Path("../data/processed")
input_path = processed_dir / "vehicle_counts.csv"
output_path = processed_dir / "traffic_density.csv"

interval_seconds = 60          # aggregation window for the time-series
lag_steps = [1, 2, 3]          # t-1, t-2, t-3 lag features
smoothing_window = 3           # rolling window used to smooth raw counts
rolling_window = 3             # rolling window for rolling mean / std / peak features


# quantile thresholds for low / moderate / high traffic categorisation, computed from the train split only
density_quantiles = [0.3, 0.6]

In [3]:
vehicle_counts = pd.read_csv(input_path)
vehicle_counts = vehicle_counts.sort_values(["scene_id", "camera_id", "timestamp_sec"])
print(f"Loaded {len(vehicle_counts)} frame-level rows from {input_path}")
vehicle_counts.head()

Loaded 11721 frame-level rows from ..\data\processed\vehicle_counts.csv


,scene_id,camera_id,split,frame_file,source_frame_idx,timestamp_sec,vehicle_count
0,S01,c001,train,frame_000000.jpg,0,0.0,1
1,S01,c001,train,frame_000001.jpg,10,1.0,0
2,S01,c001,train,frame_000002.jpg,20,2.0,1
3,S01,c001,train,frame_000003.jpg,30,3.0,2
4,S01,c001,train,frame_000004.jpg,40,4.0,1


In [4]:
def aggregate_intervals(df: pd.DataFrame, interval_seconds: int) -> pd.DataFrame:
    df = df.copy()
    df["interval_id"] = (df["timestamp_sec"] // interval_seconds).astype(int)

    aggregated = (
        df.groupby(["scene_id", "camera_id", "split", "interval_id"])
        .agg(vehicle_count=("vehicle_count", "mean"), n_frames=("vehicle_count", "count"))
        .reset_index()
    )
    aggregated["interval_start_sec"] = aggregated["interval_id"] * interval_seconds
    return aggregated.sort_values(["scene_id", "camera_id", "interval_start_sec"]).reset_index(drop=True)

In [5]:
density = aggregate_intervals(vehicle_counts, interval_seconds)
print(f"Aggregated into {len(density)} interval-level rows ({interval_seconds}s intervals)")
density.head()

Aggregated into 225 interval-level rows (60s intervals)


,scene_id,camera_id,split,interval_id,vehicle_count,n_frames,interval_start_sec
0,S01,c001,train,0,2.083333,60,0
1,S01,c001,train,1,2.766667,60,60
2,S01,c001,train,2,3.333333,60,120
3,S01,c001,train,3,2.687500,16,180
4,S01,c002,train,0,2.666667,60,0


In [6]:
def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["elapsed_time_sec"] = df["interval_start_sec"]

    clip_duration = df.groupby(["scene_id", "camera_id"])["interval_start_sec"].transform("max")
    df["relative_position"] = np.where(clip_duration > 0, df["elapsed_time_sec"] / clip_duration, 0.0)
    return df

In [7]:
density = add_temporal_features(density)
density.head()

,scene_id,camera_id,split,interval_id,vehicle_count,n_frames,interval_start_sec,elapsed_time_sec,relative_position
0,S01,c001,train,0,2.083333,60,0,0,0.000000
1,S01,c001,train,1,2.766667,60,60,60,0.333333
2,S01,c001,train,2,3.333333,60,120,120,0.666667
3,S01,c001,train,3,2.687500,16,180,180,1.000000
4,S01,c002,train,0,2.666667,60,0,0,0.000000


In [8]:
def add_smoothing(df: pd.DataFrame, smoothing_window: int) -> pd.DataFrame:
    df = df.copy()
    df["vehicle_count_smoothed"] = (
        df.groupby(["scene_id", "camera_id"])["vehicle_count"]
        .transform(lambda s: s.rolling(window=smoothing_window, min_periods=1).mean())
    )
    return df

In [9]:
density = add_smoothing(density, smoothing_window)
density.head()

,scene_id,camera_id,split,interval_id,vehicle_count,n_frames,interval_start_sec,elapsed_time_sec,relative_position,vehicle_count_smoothed
0,S01,c001,train,0,2.083333,60,0,0,0.000000,2.083333
1,S01,c001,train,1,2.766667,60,60,60,0.333333,2.425000
2,S01,c001,train,2,3.333333,60,120,120,0.666667,2.727778
3,S01,c001,train,3,2.687500,16,180,180,1.000000,2.929167
4,S01,c002,train,0,2.666667,60,0,0,0.000000,2.666667


In [10]:
def add_lag_features(df: pd.DataFrame, lag_steps: list) -> pd.DataFrame:
    df = df.copy()
    grouped = df.groupby(["scene_id", "camera_id"])["vehicle_count_smoothed"]
    for lag in lag_steps:
        df[f"lag_{lag}"] = grouped.shift(lag)
    return df

In [11]:
density = add_lag_features(density, lag_steps)
density.head()

,scene_id,camera_id,split,interval_id,vehicle_count,n_frames,interval_start_sec,elapsed_time_sec,relative_position,vehicle_count_smoothed,lag_1,lag_2,lag_3
0,S01,c001,train,0,2.083333,60,0,0,0.000000,2.083333,NaN,NaN,NaN
1,S01,c001,train,1,2.766667,60,60,60,0.333333,2.425000,2.083333,NaN,NaN
2,S01,c001,train,2,3.333333,60,120,120,0.666667,2.727778,2.425000,2.083333,NaN
3,S01,c001,train,3,2.687500,16,180,180,1.000000,2.929167,2.727778,2.425000,2.083333
4,S01,c002,train,0,2.666667,60,0,0,0.000000,2.666667,NaN,NaN,NaN


In [12]:
def add_derived_features(df: pd.DataFrame, rolling_window: int) -> pd.DataFrame:
    df = df.copy()
    grouped = df.groupby(["scene_id", "camera_id"])["vehicle_count_smoothed"]

    df["rate_of_change"] = grouped.transform(lambda s: s.diff())
    df["peak_density"] = grouped.transform(lambda s: s.rolling(window=rolling_window, min_periods=1).max())
    df["rolling_mean"] = grouped.transform(lambda s: s.rolling(window=rolling_window, min_periods=1).mean())
    df["rolling_std"] = grouped.transform(lambda s: s.rolling(window=rolling_window, min_periods=1).std())
    df["rolling_std"] = df["rolling_std"].fillna(0.0)
    return df

In [13]:
density = add_derived_features(density, rolling_window)
density.head()

,scene_id,camera_id,split,interval_id,vehicle_count,n_frames,interval_start_sec,elapsed_time_sec,relative_position,vehicle_count_smoothed,lag_1,lag_2,lag_3,rate_of_change,peak_density,rolling_mean,rolling_std
0,S01,c001,train,0,2.083333,60,0,0,0.000000,2.083333,NaN,NaN,NaN,NaN,2.083333,2.083333,0.000000
1,S01,c001,train,1,2.766667,60,60,60,0.333333,2.425000,2.083333,NaN,NaN,0.341667,2.425000,2.254167,0.241595
2,S01,c001,train,2,3.333333,60,120,120,0.666667,2.727778,2.425000,2.083333,NaN,0.302778,2.727778,2.412037,0.322418
3,S01,c001,train,3,2.687500,16,180,180,1.000000,2.929167,2.727778,2.425000,2.083333,0.201389,2.929167,2.693981,0.253777
4,S01,c002,train,0,2.666667,60,0,0,0.000000,2.666667,NaN,NaN,NaN,NaN,2.666667,2.666667,0.000000


In [14]:
def compute_density_thresholds(df: pd.DataFrame, density_quantiles: list):
    train_values = df.loc[df["split"] == "train", "vehicle_count_smoothed"]
    low_cut, high_cut = train_values.quantile(density_quantiles)
    return low_cut, high_cut

In [15]:
def categorise_density(df: pd.DataFrame, low_cut: float, high_cut: float) -> pd.DataFrame:
    df = df.copy()

    def label(value):
        if value <= low_cut:
            return "low"
        elif value <= high_cut:
            return "moderate"
        return "high"

    df["density_category"] = df["vehicle_count_smoothed"].apply(label)
    return df

In [16]:
low_cut, high_cut = compute_density_thresholds(density, density_quantiles)
density = categorise_density(density, low_cut, high_cut)

print(f"Density thresholds (from train split): low <= {low_cut:.2f} < moderate <= {high_cut:.2f} < high")
density["density_category"].value_counts()

Density thresholds (from train split): low <= 2.64 < moderate <= 3.48 < high


density_category
high        115
low          81
moderate     29
Name: count, dtype: int64

In [17]:
density.to_csv(output_path, index=False)
print(f"Saved {len(density)} rows to {output_path}")
density.head()

Saved 225 rows to ..\data\processed\traffic_density.csv


,scene_id,camera_id,split,interval_id,vehicle_count,n_frames,interval_start_sec,elapsed_time_sec,relative_position,vehicle_count_smoothed,lag_1,lag_2,lag_3,rate_of_change,peak_density,rolling_mean,rolling_std,density_category
0,S01,c001,train,0,2.083333,60,0,0,0.000000,2.083333,NaN,NaN,NaN,NaN,2.083333,2.083333,0.000000,low
1,S01,c001,train,1,2.766667,60,60,60,0.333333,2.425000,2.083333,NaN,NaN,0.341667,2.425000,2.254167,0.241595,low
2,S01,c001,train,2,3.333333,60,120,120,0.666667,2.727778,2.425000,2.083333,NaN,0.302778,2.727778,2.412037,0.322418,moderate
3,S01,c001,train,3,2.687500,16,180,180,1.000000,2.929167,2.727778,2.425000,2.083333,0.201389,2.929167,2.693981,0.253777,moderate
4,S01,c002,train,0,2.666667,60,0,0,0.000000,2.666667,NaN,NaN,NaN,NaN,2.666667,2.666667,0.000000,moderate
